# M9 — Tier 3: the real CIFAR-10 network (guide step 10)

Tiers 1 and 2 drove every solver with an **exact** noise oracle, so the only
error present was discretization error. That was deliberate — it is what made
the order fits and the crossover measurable — but it also means three of this
project's findings so far are *narrowing* the base paper's claims rather than
confirming them, and all three hand the same question to Tier 3:

| Milestone | Finding on the analytic tiers | Question for Tier 3 |
|---|---|---|
| M6 | arm A never destabilizes; `h_max` is flat in κ | does a real network destabilize it? |
| M7 | every solver keeps its textbook order under curvature | does Assumption B.1 survive a network? |
| M8 | crossover at `nfe3* ≈ 4.4–5.0` (T1), `≈ 8.9` (T2) | does it reach the paper's ~10–12 NFE? |

Tier 3 swaps the oracle for `google/ddpm-cifar10-32` (d = 3072). The reference
stops being algebra and becomes a **fine-grid run of the same checkpoint from
the same `x_T`** — so what is measured is still discretization error, not model
error (guide step 10, "Same network, same x_T").

**Scope note.** FID was cut at Gate G2, as the guide permits: L2-to-reference is
the primary read-out and this notebook ships it alone. FID is an image-quality
metric, which is exactly the standard this project argues *against* grading
solvers by.

---

### How to run this on Kaggle

1. New Notebook → **Settings → Accelerator: GPU T4 ×1**
2. **Settings → Internet: ON** (needed for `git clone` and `from_pretrained`)
3. Run All. Expect ~20 minutes including the checkpoint download.
4. Download `results/tier3_*.csv` and `figures/09_*.png` from
   `/kaggle/working/NUMERICAL-PROJECT/`, drop them into the repo, commit.

The `*.pt` reference tensors are cached to `results/` and are gitignored — they
are regenerable and large. Re-running the notebook reuses them.

In [ ]:
import os, sys, pathlib, subprocess, time

IN_KAGGLE = pathlib.Path("/kaggle").exists()

if IN_KAGGLE:
    ROOT = pathlib.Path("/kaggle/working/NUMERICAL-PROJECT")
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/MehemudAzad/NUMERICAL-PROJECT.git"],
            cwd="/kaggle/working", check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "diffusers", "accelerate"], check=True)
else:
    # Local: walk up to the repo root, same bootstrap as every other notebook.
    p = pathlib.Path.cwd()
    while not (p / "pyproject.toml").exists() and p != p.parent:
        p = p.parent
    ROOT = p

sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# NOTE: do NOT import src.arm_c here. It sets torch.set_default_dtype(float64)
# as a module-level side effect (the only way to fix the vendored solver's
# float32 timesteps without editing it -- CLAUDE.md section 8). On Tier 3 that
# would build the UNet in float64 and emit float64 solver grids.
# src.tier3.sample_dpm_solver_t3 asserts against it, so this fails loudly.
from src.grids import grid_t
from src.metrics import fit_order, l2
from src.runlog import append_row, load
from src.solvers import NFE_PER_STEP, integrate
from src.testbeds import pf_rhs_lambda, pf_rhs_t
from src.tier3 import (DiscreteSchedule, make_eps_fn, make_model_fn,
                       make_noise_schedule, sample_dpm_solver_t3)

RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"root   : {ROOT}")
print(f"device : {DEVICE}", f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")
print(f"torch  : {torch.__version__}, default dtype {torch.get_default_dtype()}")
assert torch.get_default_dtype() == torch.float32, "restart the kernel: something set float64"
if DEVICE == "cpu":
    print("\n!! No GPU. This notebook will run, but slowly -- set Accelerator to T4.")

## 1. The checkpoint, the schedule, and the fixed starting noise

Two `NoiseScheduleVP` objects are built from the **same** betas, on purpose:

- **float32, on GPU** — for `DPM_Solver` and `model_wrapper` (arm C). `DPM_Solver`
  multiplies schedule coefficients straight into the state, so a float64
  schedule would promote the float32 batch and then fail inside the float32 UNet.
- **float64, on CPU** — wrapped in `DiscreteSchedule` for arms A and B. Their
  `f(t)` comes from a central difference of λ, which is hopeless in float32
  (λ ≈ 5, dt = 1e-5, so float32 rounding alone costs ~0.03 in the derivative).

They differ only by float32 rounding (~1e-7), four orders of magnitude below
Tier 3's ~1e-3 floor — so all three arms still share one schedule, which is the
property Gate G1 established for Tiers 1–2.

In [ ]:
from diffusers import DDPMScheduler, UNet2DModel

MODEL_ID = "google/ddpm-cifar10-32"
SEED = 0
N_SAMPLES = 64          # a knob. 64 is ample for an L2 curve and leaves T4 headroom.

t0 = time.time()
unet = UNet2DModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
ddpm = DDPMScheduler.from_pretrained(MODEL_ID)
print(f"checkpoint loaded in {time.time()-t0:.1f}s  "
      f"({sum(p.numel() for p in unet.parameters())/1e6:.1f}M params)")

betas = ddpm.betas.double()
ns32 = make_noise_schedule(betas, dtype=torch.float32, device=DEVICE)  # arm C
ns64 = make_noise_schedule(betas, dtype=torch.float64)                 # arms A/B (CPU)
ds = DiscreteSchedule(ns64)

# `.sample` matters: UNet2DModel returns a dataclass, not a tensor.
model_fn = make_model_fn(unet, ns32)
eps_fn = make_eps_fn(model_fn)

torch.manual_seed(SEED)
x_T = torch.randn(N_SAMPLES, 3, 32, 32, device=DEVICE)

T, T_END = 1.0, ds.t_min
print(f"betas  : {betas[0]:.2e} -> {betas[-1]:.4f}, N = {len(betas)}")
print(f"x_T    : {tuple(x_T.shape)} {x_T.dtype}, seed {SEED}")
print(f"range  : t from {T} down to {T_END:g}")

## 2. Schedule sanity — the Tier-3 analogue of M4's agreement check

M4 verified that `src/schedule.py` and the vendored `NoiseScheduleVP('linear')`
agree to 1e-11, which is what let arms A/B and arm C be compared on Tiers 1–2.
Tier 3 cannot make that claim: the checkpoint's schedule is the **discretised**
VP-linear one, and the two genuinely differ. This cell measures by how much —
and that measurement is the justification for deriving arms A/B from the
checkpoint's own schedule rather than reusing `src/schedule.py`.

Two things are asserted rather than printed:

- **`ns.total_N == 1000`.** `numerical_clip_alpha` silently truncates the beta
  array if λ(T) < −5.1. Here λ(T) ≈ −5.06, *just* inside — and a clip would move
  `t_0 = 1/total_N` without saying so.
- **λ strictly decreasing.** The project's most common bug, re-checked on a
  schedule none of the earlier milestones touched.

In [ ]:
from src import schedule as cont

assert ns64.total_N == 1000, f"beta array was clipped to {ns64.total_N} -- t_0 has moved"

tt = np.linspace(T_END, T, 500)
d_lam = np.abs(ds.lmbda(tt) - cont.lmbda(tt))
d_f = np.abs(ds.f(tt) - cont.f(tt)) / np.abs(cont.f(tt))

assert np.all(np.diff(ds.lmbda(tt)) < 0), "lambda must be strictly decreasing in t"
assert np.allclose(ds.alpha(tt) ** 2 + ds.sigma(tt) ** 2, 1.0, atol=1e-12)

print(f"lambda span            : {ds.lmbda(T):.4f} -> {ds.lmbda(T_END):.4f}")
print(f"max |lam_disc - lam_cont| = {d_lam.max():.3e}   (at t = {tt[d_lam.argmax()]:.4f})")
print(f"max relative |df/f|       = {d_f.max():.3e}   (at t = {tt[d_f.argmax()]:.4f})")
print()
print("The discrete and continuous schedules are the same schedule, one sampled.")
print("They are NOT interchangeable: reusing src/schedule.py for arms A/B would")
print(f"put a systematic ~{100*d_f.max():.1f}% error in f under every Tier-3 curve.")
print("Hence DiscreteSchedule -- all three arms read one schedule, the checkpoint's own.")

## 3. The reference trajectories, cached

The guide's second "costs you a day" gotcha: *recomputing 200-NFE trajectories
is the single biggest time sink in this project.* Every reference below is
written to `results/tier3_ref_*.pt` and reloaded on a re-run.

Five references, serving three different purposes:

| tag | steps | `t_end` | purpose |
|---|---|---|---|
| `s201_e1e-3` | 201 | 1e-3 | **the** reference — every error in §5 is measured against it |
| `s300_e1e-3` | 300 | 1e-3 | reference *uncertainty*: ‖s300 − s201‖ is the floor below which no error is meaningful |
| `s201_e2e-3` / `s201_e5e-3` / `s201_e1e-2` | 201 | 2e-3 … 1e-2 | the `t_end` truncation term for §7 |

**A constraint worth stating.** The discrete schedule tabulates `t` on
`linspace(0,1,1001)[1:]`, so `t = 1/N = 1e-3` is its **floor** — below it,
`inverse_lambda` extrapolates off the table. The guide's ε = 1e-4 row of Table 6
is therefore not reachable with this checkpoint, and the truncation study sweeps
`t_end` *down toward* the floor instead of past it.

In [ ]:
def cached_reference(tag, steps, t_end, order=3):
    # Fine-grid DPM-Solver-3 reference, computed once and cached to disk.
    path = RESULTS / f"tier3_ref_{tag}.pt"
    if path.exists():
        x = torch.load(path, map_location=DEVICE)
        print(f"  {tag:<12s} loaded from cache")
        return x
    t0 = time.time()
    x, nfe = sample_dpm_solver_t3(model_fn, ns32, x_T, T, t_end, order, steps)
    torch.save(x.cpu(), path)
    print(f"  {tag:<12s} {nfe:4d} NFE in {time.time()-t0:6.1f}s")
    return x


print("references:")
ref = cached_reference("s201_e1e-3", steps=201, t_end=1e-3)     # the reference
ref_fine = cached_reference("s300_e1e-3", steps=300, t_end=1e-3)  # its uncertainty
ref_te = {te: cached_reference(f"s201_e{te:g}", steps=201, t_end=te)
          for te in [2e-3, 5e-3, 1e-2]}


def err_to(x, r):
    # L2 distance to a reference, via the project's shared metric.
    return l2(x.detach().cpu().numpy().ravel(), r.detach().cpu().numpy().ravel())


FLOOR = err_to(ref_fine, ref)
print(f"\nreference uncertainty ||ref_300 - ref_201|| = {FLOOR:.4e}"
      f"   ({FLOOR/np.sqrt(N_SAMPLES):.3e} per sample)")
print("No error measured below this line means anything -- it is float32 plus the")
print("network's own non-smoothness, not discretization error. Labelled on every figure.")

## 4. The sweep

Nine curves — the full three-arm design, on a real network:

- **arm A** `euler, midpoint, rk4` on a grid uniform in `t` (`grid_t`)
- **arm B** the same three on a grid uniform in λ
- **arm C** `dpm1, dpm2, dpm3` via `singlestep_fixed`, `skip_type="logSNR"`

Arms A and B march through **`src.solvers.integrate`** — the same integrator,
unmodified, that marched Tiers 1 and 2; only the state is now a float32 CUDA
tensor and the oracle is a UNet. `heun3` and `ab2` are dropped from Tier 3 (the
guide's step-10 budget cut, seven solvers → four-ish): `heun3` duplicates arm C's
order-3 slot and `ab2` duplicates `midpoint`'s order-2 one.

Budgets are set in **NFE**, not steps, and each solver's step count is chosen so
its *measured* NFE lands on the budget. Arm C's `steps` is rounded down to a
multiple of `order`, since `singlestep_fixed` spends `(steps // order) * order`.

In [ ]:
NFE_BUDGETS = [10, 12, 15, 20, 30, 50, 80, 120]
ARM_AB = {"euler": 1, "midpoint": 2, "rk4": 4}     # solver -> NFE per step
ARM_C = {"dpm1": 1, "dpm2": 2, "dpm3": 3}          # solver -> order

LAM_SPAN = abs(ds.lmbda(T) - ds.lmbda(T_END))

def h_t(n):
    return abs(T - T_END) / n

def h_lam(n):
    return LAM_SPAN / n

plan = []
for b in NFE_BUDGETS:
    for s, per in ARM_AB.items():
        n = max(1, b // per)
        plan.append(("AB", s, n, n * per))
    for s, order in ARM_C.items():
        steps = max(order, (b // order) * order)
        plan.append(("C", s, steps, steps))

total_nfe = 2 * sum(p[3] for p in plan if p[0] == "AB") + sum(p[3] for p in plan if p[0] == "C")
print(f"{len(NFE_BUDGETS)} budgets x 9 curves = {2*len([p for p in plan if p[0]=='AB']) + len([p for p in plan if p[0]=='C'])} runs")
print(f"total network calls in the sweep: {total_nfe}")

In [ ]:
CSV = RESULTS / "tier3_error.csv"
CSV.unlink(missing_ok=True)

sweep_t0 = time.time()

# -- arms A and B: the shared integrator, the network as the oracle ---------
for arm, rhs, gridf, hf in [
    ("A", lambda x, t: pf_rhs_t(eps_fn, x, t, sched=ds), lambda n: grid_t(T, T_END, n), h_t),
    ("B", lambda x, l: pf_rhs_lambda(eps_fn, x, l, sched=ds),
     lambda n: np.linspace(ds.lmbda(T), ds.lmbda(T_END), n + 1), h_lam),
]:
    for solver, per in ARM_AB.items():
        order = {"euler": 1, "midpoint": 2, "rk4": 4}[solver]
        for b in NFE_BUDGETS:
            n = max(1, b // per)
            xf, nfe = integrate(rhs, x_T, gridf(n), solver)
            div = bool(np.isnan(nfe))
            if not div:
                assert nfe == NFE_PER_STEP[solver] * n, (solver, n, nfe)
            append_row(str(CSV), tier=3, testbed="cifar10", arm=arm, solver=solver,
                       order=order, kappa=np.nan, h=hf(n), nfe=nfe,
                       err_l2=np.nan if div else err_to(xf, ref),
                       diverged=div, seed=SEED)
        print(f"  arm {arm} {solver:<9s} done  [{time.time()-sweep_t0:6.1f}s]")

# -- arm C: the authors' code, singlestep_fixed -----------------------------
for solver, order in ARM_C.items():
    for b in NFE_BUDGETS:
        steps = max(order, (b // order) * order)
        xf, nfe = sample_dpm_solver_t3(model_fn, ns32, x_T, T, T_END, order, steps)
        assert nfe == (steps // order) * order
        append_row(str(CSV), tier=3, testbed="cifar10", arm="C", solver=solver,
                   order=order, kappa=np.nan, h=h_lam(steps // order), nfe=nfe,
                   err_l2=err_to(xf, ref), diverged=False, seed=SEED)
    print(f"  arm C {solver:<9s} done  [{time.time()-sweep_t0:6.1f}s]")

df = load(str(CSV))
print(f"\n{len(df)} rows -> {CSV.relative_to(ROOT)}   ({time.time()-sweep_t0:.1f}s total)")
df.head()

## 5. Error vs NFE — all three arms, with the floor drawn

The cost-normalised comparison: every arm on one axis, plotted against
**measured** NFE, never step count. The dashed horizontal line is the reference's
own uncertainty from §3 — the level below which a point measures float32 and
network non-smoothness rather than discretization error.

In [ ]:
ARM_STYLE = {"A": ("tab:blue", "-o"), "B": ("tab:orange", "--s"), "C": ("tab:green", "-^")}
ARM_LABEL = {"A": "arm A (raw ODE in t)", "B": "arm B (lambda-ODE)", "C": "arm C (DPM-Solver)"}

fig, ax = plt.subplots(figsize=(8.5, 6))
for (arm, solver), g in df.groupby(["arm", "solver"]):
    g = g.sort_values("nfe")
    color, style = ARM_STYLE[arm]
    ax.loglog(g["nfe"], g["err_l2"], style, color=color, ms=4, alpha=0.85,
              label=f"{arm}: {solver}")

ax.axhline(FLOOR, color="tab:red", ls="--", lw=1.3)
ax.text(ax.get_xlim()[1], FLOOR, " reference floor\n ||ref_300 - ref_201||",
        color="tab:red", fontsize=8, va="bottom", ha="right")
ax.axhspan(ax.get_ylim()[0], FLOOR, color="tab:red", alpha=0.06)

ax.set_xlabel("NFE (measured network calls)")
ax.set_ylabel("L2 error to the 201-NFE reference")
ax.set_title(f"Tier 3, {MODEL_ID}: error vs NFE, all three arms (n={N_SAMPLES})")
ax.legend(fontsize=8, ncol=3, loc="lower left")
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "09_tier3_error_vs_nfe.png", dpi=150)
plt.show()

## 6. The order table — does Assumption B.1 survive a real network?

This is M7's question, asked of a network instead of an analytic score. The
paper's order theorem rests on **Assumption B.1**: that the model's derivatives
with respect to λ exist and are continuous up to order *k*+1. Tiers 1 and 2
satisfy it exactly and every solver hit its textbook order there (M7: worst
deviation 0.084). A neural network has no obligation to satisfy it.

`fit_order`'s sliding window should exclude the flat floor on its own — the cell
below reports each fit window so that can be checked rather than assumed, exactly
as M5 and M7 do.

**Read these slopes with two caveats, both of them expected.** First, M5's lesson
(CLAUDE.md §6): `fit_order` is not immune to a badly-chosen sweep range, and on
Tier 3 the range is not free — 10–120 NFE is the practically interesting band and
also, for the higher-order arm-A/B methods, squarely **pre-asymptotic**. A slope
well under theory here is a statement about that band, not a refutation of the
order theorem. Second, the reference is itself a 201-NFE run, so a sweep point at
120 NFE is only ~1.7× coarser than its own yardstick; that is what the floor line
measures and why it is drawn on every figure. `hit_floor` marks the curves with no
asymptotic region left to fit.

In [ ]:
rows = []
for (arm, solver), g in df.groupby(["arm", "solver"]):
    g = g.sort_values("h", ascending=False)
    fit = fit_order(g["h"].to_numpy(), g["err_l2"].to_numpy())
    rows.append(dict(arm=arm, solver=solver,
                     theoretical_order=int(g["order"].iloc[0]),
                     measured_slope=fit["slope"], r2=fit["r2"],
                     window=fit["window"], n_in_window=fit["n"], n_swept=len(g),
                     min_err=g["err_l2"].min()))

order_table = pd.DataFrame(rows).sort_values(["arm", "solver"]).reset_index(drop=True)
order_table["gap_vs_theory"] = order_table["measured_slope"] - order_table["theoretical_order"]
order_table["hit_floor"] = order_table["min_err"] < 3 * FLOOR
pd.set_option("display.width", 140)
order_table

In [ ]:
t12 = []
for tier, name in [(1, "tier1_order.csv"), (2, "tier2_order.csv")]:
    p = RESULTS / name
    if p.exists():
        d = load(str(p))
        for (arm, solver), g in d.groupby(["arm", "solver"]):
            g = g.sort_values("h", ascending=False)
            t12.append(dict(tier=tier, arm=arm, solver=solver,
                            slope=fit_order(g["h"].to_numpy(), g["err_l2"].to_numpy())["slope"]))
prev = pd.DataFrame(t12)

print("Tier 3 measured slope vs the analytic tiers, per (arm, solver):\n")
cmp = order_table[["arm", "solver", "theoretical_order", "measured_slope"]].rename(
    columns={"measured_slope": "tier3"})
for tier in (1, 2):
    if len(prev):
        s = prev[prev["tier"] == tier].set_index(["arm", "solver"])["slope"]
        cmp[f"tier{tier}"] = [s.get((a, sv), np.nan) for a, sv in zip(cmp["arm"], cmp["solver"])]
print(cmp.to_string(index=False))

worst = (order_table["gap_vs_theory"]).abs().max()
print(f"\nworst |measured - theoretical| on Tier 3 : {worst:.3f}")
print(f"  (M7 reported 0.084 on Tier 2, 0.059 shift from Tier 1)")
print(f"curves that reached the reference floor  : "
      f"{order_table['hit_floor'].sum()} of {len(order_table)}")

## 7. The three-way error decomposition

The guide, step 10: *total error on Tier 3 has three components, not one —
discretization error, truncation from stopping at `t_end` instead of 0, and
network approximation error. Your deck discusses only the first.*

Each is measured here, and each is measurable *without* knowing the true score:

| component | how it is measured |
|---|---|
| **discretization** | the swept curve: ‖x_N − ref‖ at each NFE |
| **`t_end` truncation** | ‖ref(t_end) − ref(1e-3)‖ — how far the endpoint moves by stopping early, same solver, same grid density |
| **network + precision floor** | ‖ref_300 − ref_201‖ — what two supposedly-converged references disagree by; float32 plus the network's own non-smoothness |

Read off the figure: the NFE past which the discretization curve drops *below*
the other two is the point where spending more network calls stops buying
accuracy. That is the mechanism behind the paper's own Table 4 oddity —
4.39 FID at 15 NFE becoming **5.52 at 20 NFE**, more calls and worse images.

In [ ]:
dec_rows = [dict(component="network+precision floor", setting="||ref_300 - ref_201||",
                 value=FLOOR)]
for te, r in ref_te.items():
    dec_rows.append(dict(component="t_end truncation", setting=f"t_end={te:g} vs 1e-3",
                         value=err_to(r, ref)))

best = df[df["arm"] == "C"].sort_values("nfe")
for _, r in best[best["solver"] == "dpm3"].iterrows():
    dec_rows.append(dict(component="discretization (arm C, dpm3)",
                         setting=f"nfe={int(r['nfe'])}", value=r["err_l2"]))

dec = pd.DataFrame(dec_rows)
dec.to_csv(RESULTS / "tier3_decomposition.csv", index=False)
print(dec.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6))

for solver, color in [("dpm1", "tab:blue"), ("dpm2", "tab:purple"), ("dpm3", "tab:green")]:
    g = df[(df["arm"] == "C") & (df["solver"] == solver)].sort_values("nfe")
    ax.loglog(g["nfe"], g["err_l2"], "-o", color=color, ms=4,
              label=f"discretization ({solver})")

ax.axhline(FLOOR, color="tab:red", ls="--", lw=1.4, label="network + precision floor")
for (te, r), ls in zip(ref_te.items(), [":", "-.", (0, (3, 1, 1, 1))]):
    v = err_to(r, ref)
    ax.axhline(v, color="tab:brown", ls=ls, lw=1.2,
               label=f"t_end truncation (stop at {te:g})")

# where discretization drops under the floor
g3 = df[(df["arm"] == "C") & (df["solver"] == "dpm3")].sort_values("nfe")
under = g3[g3["err_l2"] < FLOOR]
if len(under):
    n_star = int(under["nfe"].iloc[0])
    ax.axvline(n_star, color="k", ls=":", lw=1)
    ax.text(n_star, ax.get_ylim()[1], f" dpm3 hits the floor\n at {n_star} NFE",
            fontsize=8, va="top")

ax.set_xlabel("NFE")
ax.set_ylabel("L2 distance")
ax.set_title("Tier 3: the three error components, on one axis")
ax.legend(fontsize=8, loc="lower left")
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "09_tier3_decomposition.png", dpi=150)
plt.show()

## 8. What Tier 3 says about M6, M7 and M8

Fill these three in from the numbers above — they are the questions the earlier
milestones explicitly deferred, and they are what the report leads with.

**M7's question — does the order survive a real network?** Compare the §6 table's
`gap_vs_theory` column against M7's 0.084. The guide predicts (Part 5) that
*"Tier 3 slopes come out lower than theory and flatten early"*; the `hit_floor`
column says which curves reached the reference floor and therefore have no
asymptotic region left to fit.

**M6's question — does arm A destabilize?** M6 found `h_max` exactly flat on
Tier 1 and attributed it to a cancellation specific to the linear, decoupled
problem. Check the `diverged` column of `tier3_error.csv`: arm A blowing up at
the coarse budgets where arms B and C do not is what §4.2 of the paper claims and
what Tier 1 refused to show.

**M8's question — where is the crossover?** Compare `dpm1` and `dpm3` in the §5
figure at matched NFE. M8 measured `nfe3* ≈ 4.4–5.0` (Tier 1) and `≈ 8.9`
(Tier 2, curvature) against the paper's ~10–12. Tier 3 has the two ingredients
both analytic tiers lack — network approximation error, and d = 3072 instead of
d = 2–3.

**On the framing.** Whatever these come out as, they should be reported the way
M6, M7 and M8 were: as *narrowing* the paper's claims where the evidence narrows
them, not as confirmation. Three tiers agreeing is a result; three tiers
disagreeing in an explainable way is a better one. M10 turns this into the
predictions ledger.

## 9. Validation — run the unit tests

In [ ]:
out = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_tier3.py", "-q"],
    cwd=ROOT, capture_output=True, text=True,
)
print(out.stdout); print(out.stderr)
assert out.returncode == 0, "M9 Tier-3 tests failed"

for f in ["results/tier3_error.csv", "results/tier3_decomposition.csv",
          "figures/09_tier3_error_vs_nfe.png", "figures/09_tier3_decomposition.png"]:
    assert (ROOT / f).exists(), f"missing deliverable: {f}"
    print(f"  ok  {f}")

print("\nM9 validation: PASS")
print("\nDownload results/tier3_*.csv and figures/09_*.png back into the repo.")